[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cstecker/politicsRLab/blob/main/In%20welcher%20Blase%20bin%20ich%20denn%20hier%20gelandet%20-%202026-10.ipynb)

# In welcher Blase bin ich denn hier gelandet?

**Kursumfrage BRD · Wintersemester 2026/27**

Wer sitzt mit uns im Hörsaal? Welche politischen Positionen sind häufig, welche selten? Wir untersuchen unsere eigene Kursumfrage und lernen dabei, **Häufigkeiten, Mittelwert, Modalwert und Median(kategorien)** auseinanderzuhalten.

Sie brauchen keine R-Vorkenntnisse. Führen Sie die Zellen der Reihe nach mit dem Play-Button oder **Strg + Enter** aus. Speichern Sie für eigene Änderungen eine Kopie des Notebooks. Wenn Sie eine frühe Zelle ändern, führen Sie auch die folgenden Zellen erneut aus.

**Vor dem ersten Ergebnis:** Vermuten Sie, dass sich die Antworten eher links, in der Mitte oder rechts häufen? Begründen Sie Ihre Erwartung in einem Satz. Es geht um die Antworten im Kurs, nicht um Ihre persönliche Position.

## 1. R vorbereiten

Das Notebook verwendet **R**. Das Paket **tidyverse** enthält Werkzeuge zum Einlesen, Zählen und Zeichnen. Falls es in der Laufzeit fehlt, installiert die erste Zelle es automatisch; das kann beim ersten Start einige Minuten dauern.

In [ ]:
if (!requireNamespace("tidyverse", quietly = TRUE)) {
  install.packages("tidyverse", repos = "https://cloud.r-project.org")
}
library(tidyverse)
options(repr.plot.width = 10, repr.plot.height = 5)


## 2. Daten laden und verstehen

Die Übungsdatei enthält ausschließlich **abgeschlossene Interviews mit Einwilligung**. Persönliche Codes, Antwort-IDs, Zeitstempel, Geburtsorte und offene Antworten wurden entfernt. Seltene Wahlabsichten werden zusammengefasst. Geschlechtsangaben außerhalb der beiden ausreichend großen Gruppen werden als fehlend gespeichert; eine Gruppe mit weniger als fünf Fällen wird ebenfalls unterdrückt. Die Zeilen sind gemischt.

Jede Zeile ist ein Interview, jede Spalte eine Variable. `NA` heißt: Für diese Variable steht kein veröffentlichter Wert zur Verfügung. Bei den Positionsskalen bedeutet das eine übersprungene Frage; bei Geschlecht und Wahlabsicht kann es zusätzlich eine unterdrückte Angabe sein.

Die nächste Zelle lädt die veröffentlichte Datei. Wenn Sie eine neuere Übungsdatei von der Lehrperson erhalten haben, laden Sie diese vorher über die Dateiansicht in die Colab-Laufzeit hoch. Unter dem gleichen Dateinamen wird dann Ihre hochgeladene Datei verwendet. Laden Sie ausschließlich die bereitgestellte **Colab-Datei**, keinen Rohexport.

In [ ]:
datei <- "survey2026-10_colab.rds"
if (!file.exists(datei)) {
  tryCatch(
    download.file(
      "https://raw.githubusercontent.com/cstecker/politicsRLab/main/data/survey2026-10_colab.rds",
      datei, mode = "wb", quiet = TRUE
    ),
    error = function(e) {
      if (file.exists(datei)) unlink(datei)
      stop("Die Übungsdatei ist online nicht erreichbar. Laden Sie die bereitgestellte survey2026-10_colab.rds über die Dateiansicht hoch und starten Sie diese Zelle erneut.", call. = FALSE)
    }
  )
}
survey <- readRDS(datei)
stopifnot(
  identical(names(survey), c("geschlecht", "vote", "lire_self", "econ_self",
                            "immi_self", "klim_self", "euin_self")),
  all(is.na(survey$lire_self) | survey$lire_self %in% 1:11)
)
stand <- attr(survey, "kursumfrage")$snapshot_date
cat("Datenstand:", stand, "· Interviews:", nrow(survey), "\n")
glimpse(survey)


| Variable | Bedeutung / Skalenenden |
|---|---|
| `geschlecht` | Weiblich / Männlich; weitere oder seltene Angaben: NA |
| `vote` | Wahlabsicht; seltene Angaben zusammengefasst |
| `lire_self` | 1 = links, 11 = rechts |
| `econ_self` | 1 = mehr Sozialstaat, 11 = weniger Sozialstaat |
| `immi_self` | 1 = Zuzug erleichtern, 11 = Zuzug einschränken |
| `klim_self` | 1 = Vorrang Klimaschutz, 11 = Vorrang Wirtschaftswachstum |
| `euin_self` | 1 = mehr EU-Kompetenzen, 11 = weniger EU-Kompetenzen |

Die Skalen bleiben bei **1 bis 11**. Die Mitte ist 6. `head()` zeigt die ersten sechs Zeilen; `count()` zählt Antworten. Die Zeichen `|>` heißen Pipe: „Nimm das Ergebnis links und gib es an den nächsten Befehl weiter.“

In [ ]:
head(survey)
survey |> count(geschlecht, name = "Anzahl")
survey |> count(immi_self, name = "Anzahl")


## 3. Wie würden die Teilnehmenden wählen?

Zuerst zählen wir die Wahlabsichten. Die Tabelle enthält auch fehlende Werte. Für die **Prozentanteile** verwenden wir anschließend nur veröffentlichte Wahlabsichten und nennen den Nenner ausdrücklich. „Nichtwahl“ ist eine inhaltliche Antwort und bleibt enthalten.

In [ ]:
survey |> count(vote, sort = TRUE)
n_vote <- sum(!is.na(survey$vote))
cat("Veröffentlichte Wahlabsichten:", n_vote,
    "· fehlend/unterdrückt:", sum(is.na(survey$vote)), "\n")
vote_share <- survey |>
  filter(!is.na(vote)) |>
  count(vote, sort = TRUE) |>
  mutate(share = n / sum(n), Prozent = round(100 * share, 1))
vote_share


In [ ]:
if (n_vote > 0) {
  p_vote <- vote_share |>
    mutate(vote = forcats::fct_reorder(vote, share)) |>
    ggplot(aes(x = vote, y = share)) +
    geom_col(fill = "#9C1C26", width = 0.7) +
    coord_flip() +
    scale_y_continuous(labels = scales::label_percent(accuracy = 1)) +
    labs(title = "Wahlabsicht in der Kursumfrage",
         subtitle = paste("Datenstand", stand), x = NULL,
         y = "Anteil der veröffentlichten Wahlabsichten",
         caption = paste("n =", n_vote, "· einschließlich Nichtwahl; seltene Angaben zusammengefasst.")) +
    theme_minimal(base_size = 13)
  print(p_vote)
} else {
  cat("Für die Wahlabsicht sind keine auswertbaren Angaben vorhanden.\n")
}


## 4. Links oder rechts: erst die Verteilung, dann die Kennzahl

`lire_self` ist unsere Links-Rechts-Selbsteinstufung. **Vorhersage:** Welche Kategorie wird am häufigsten vorkommen? Werden Mittelwert und Median gleich sein?

Wir schließen fehlende Antworten **nur für diese Analyse** aus. Alle elf Kategorien bleiben in der Häufigkeitstabelle sichtbar, auch solche ohne Antwort. Der kumulierte Anteil addiert die Anteile von links nach rechts.

In [ ]:
n_fehlend <- sum(is.na(survey$lire_self))
x <- sort(survey$lire_self[!is.na(survey$lire_self)])
n_gueltig <- length(x)
if (n_gueltig == 0) stop("Keine gültige Links-Rechts-Antwort; diese Analyse ist nicht möglich.")
haeufigkeiten <- tibble(lire_self = x) |>
  count(lire_self) |>
  complete(lire_self = 1:11, fill = list(n = 0L)) |>
  arrange(lire_self) |>
  mutate(Anteil = n / sum(n), kumuliert = cumsum(Anteil))
cat("Gültige Antworten:", n_gueltig, "· fehlend:", n_fehlend, "\n")
haeufigkeiten


### Mittelwert: alle Werte gehen ein

Wir addieren die Antworten und teilen durch ihre Anzahl. `mean()` erledigt dieselbe Rechnung. Der Mittelwert kann zwischen Kategorien liegen.

Die Berechnung setzt voraus, dass die Abstände zwischen den elf Stufen als gleich groß behandelt werden. Das ist eine zusätzliche Annahme über diese **geordnete Selbsteinstufung**, keine Eigenschaft, die die Zahlen allein garantieren.

In [ ]:
mittelwert <- mean(x)
tibble(Summe = sum(x), Anzahl = n_gueltig,
       Mittelwert = mittelwert, Kontrolle = sum(x) / n_gueltig)


### Modalwert: die häufigste Kategorie

Der Modalwert braucht keine Annahme über gleiche Abstände. Wir suchen die Kategorie mit den meisten Antworten. Bei Gleichstand gibt es **mehrere Modalwerte**; wir behalten sie alle. R kennt zwar einen Befehl `mode()`, dieser liefert aber den Speichertyp eines Objekts und berechnet keinen statistischen Modalwert.

In [ ]:
modalwerte <- haeufigkeiten |>
  filter(n == max(n)) |>
  pull(lire_self)
modalwerte


### Median(kategorien): die Mitte der geordneten Antworten

Sortieren Sie die Antworten von links nach rechts. Bei ungerader Fallzahl gibt es **einen mittleren Rangplatz**, bei gerader Fallzahl **zwei**. Entscheidend ist zunächst, in welchen Kategorien diese Rangplätze liegen.

Liegen beide in derselben Kategorie, ist diese die Mediankategorie. Liegen sie in verschiedenen Kategorien, geben wir den **Medianbereich** von der unteren bis zur oberen mittleren Kategorie an: Jede Kategorie in diesem Bereich erfüllt die Bedingung, dass mindestens die Hälfte der Antworten auf oder unter und mindestens die Hälfte auf oder über ihr liegt. `median()` bildet dagegen bei gerader Fallzahl das arithmetische Mittel der beiden mittleren Zahlen. Dieser Zahlenwert muss keine tatsächlich gewählte Kategorie sein.

In [ ]:
rang_mitte <- unique(c(floor((n_gueltig + 1) / 2), ceiling((n_gueltig + 1) / 2)))
mittlere_kategorien <- x[rang_mitte]
median_unten <- min(mittlere_kategorien)
median_oben <- max(mittlere_kategorien)
median_kategorien <- seq.int(median_unten, median_oben)
median_text <- if (median_unten == median_oben) {
  as.character(median_unten)
} else {
  paste(median_unten, "bis", median_oben)
}
tibble(Rangplatz = rang_mitte, Kategorie = mittlere_kategorien)
tibble(Mediankategorien = median_text, numerischer_Median = median(x))


### Drei Antworten auf drei verschiedene Fragen

Vergleichen Sie Ihre Vorhersage mit der Ergebnistabelle. Eine Kennzahl verdichtet die Verteilung; sie beschreibt nicht jede einzelne Person.

In [ ]:
fmt <- function(z) format(round(z, 2), nsmall = 2, decimal.mark = ",", trim = TRUE)
kennzahlen <- tibble(
  Maß = c("Mittelwert", "Modalwert(e)", "Mediankategorie(n)"),
  Ergebnis = c(fmt(mittelwert), paste(modalwerte, collapse = ", "), median_text),
  Frage = c("Wo liegt der rechnerische Durchschnitt?",
            "Welche Kategorie(n) wurde(n) am häufigsten gewählt?",
            "Wo liegt die Mitte der geordneten Antworten?")
)
kennzahlen


### Alle drei Maße in einer Abbildung

Alle drei Teilgrafiken zeigen dieselbe Antwortverteilung mit **gleicher Achsenskalierung**. Oben markiert die Linie den Mittelwert, in der Mitte markieren die Linien alle Modalwerte. Unten zeigt das Band die Mediankategorie beziehungsweise den Medianbereich. So bleiben die Maße auch sichtbar, wenn Modalwert und Median zusammenfallen.

In [ ]:
titel_mittel <- paste0("Mittelwert = ", fmt(mittelwert))
titel_modal <- paste0("Modalwert(e) = ", paste(modalwerte, collapse = ", "))
titel_median <- paste0("Mediankategorie(n) = ", median_text)
panel_labels <- c(titel_mittel, titel_modal, titel_median)
plot_daten <- bind_rows(lapply(panel_labels, function(label) {
  haeufigkeiten |> mutate(Maß = label)
})) |>
  mutate(Maß = factor(Maß, levels = panel_labels))
linien <- tibble(Maß = factor(c(titel_mittel, rep(titel_modal, length(modalwerte))),
                             levels = panel_labels),
                 Wert = c(mittelwert, modalwerte))
band <- tibble(Maß = factor(titel_median, levels = panel_labels),
               unten = median_unten - 0.42, oben = median_oben + 0.42)
p_lage <- ggplot(plot_daten, aes(x = lire_self, y = n)) +
  geom_rect(data = band, aes(xmin = unten, xmax = oben, ymin = 0, ymax = Inf),
            inherit.aes = FALSE, fill = "#E69F00", alpha = 0.25) +
  geom_col(fill = "#657A86", width = 0.7) +
  geom_vline(data = linien, aes(xintercept = Wert),
             color = "#9C1C26", linewidth = 0.9, linetype = "dashed") +
  geom_text(aes(label = n), vjust = -0.4, size = 3.6) +
  facet_wrap(vars(Maß), ncol = 1) +
  scale_x_continuous(breaks = 1:11, limits = c(0.5, 11.5)) +
  scale_y_continuous(breaks = function(limits) unique(round(scales::breaks_pretty(n = 4)(limits))),
                     expand = expansion(mult = c(0, 0.18))) +
  labs(title = "Links-Rechts-Selbsteinstufung: drei Lageparameter",
       subtitle = paste0("Datenstand ", stand, " · ", n_gueltig,
                         " gültige Antworten · ", n_fehlend, " fehlende Antworten"),
       x = "Links-Rechts-Skala: 1 = links, 6 = Mitte, 11 = rechts",
       y = "Anzahl Antworten",
       caption = "Linien: Mittelwert und Modalwert(e). Band: Mediankategorie(n).
Der Mittelwert behandelt die Abstände zwischen den Stufen als gleich groß.") +
  theme_minimal(base_size = 13) +
  theme(panel.grid.minor = element_blank(), panel.grid.major.x = element_blank(),
        strip.text = element_text(face = "bold", hjust = 0),
        plot.margin = margin(12, 16, 12, 12))
options(repr.plot.width = 10, repr.plot.height = 9)
print(p_lage)


**Erklären Sie den Befund:**

1. Warum können Mittelwert und Median voneinander abweichen?
2. Was würden wir übersehen, wenn wir nur den Mittelwert sähen?
3. Welche Kennzahl könnten Sie auch für reine Parteinamen berechnen? Warum die anderen nicht?
4. Beschreibt diese Umfrage alle Studierenden oder die Bevölkerung Deutschlands? Begründen Sie Ihre Antwort.

Die politische Mitte einer Skala (hier 6) ist etwas anderes als der Median unserer Antworten. Ob der Kurs im Vergleich zur Bevölkerung eine „Blase“ ist, braucht außerdem eine passende Vergleichsstichprobe mit derselben Frage.

## 5. Kleine Übung: Welche Regel passt?

Berechnen Sie **erst selbst**, dann führen Sie die Zelle aus. Die Zahlen stehen für Kategorien derselben Links-Rechts-Skala.

| Fall | Antworten | Ihre Aufgabe |
|---|---|---|
| A | 1, 2, 2, 3, 11 | Mittelwert, Modalwert und Mediankategorie bestimmen |
| B | 2, 2, 5, 5 | Modalwerte und Medianbereich bestimmen |
| C | 3, 3, NA | Fehlende Werte behandeln und die drei Maße bestimmen |

In [ ]:
beispiele <- list(A = c(1, 2, 2, 3, 11), B = c(2, 2, 5, 5), C = c(3, 3, NA))
for (fall in names(beispiele)) {
  z <- sort(beispiele[[fall]][!is.na(beispiele[[fall]])])
  hz <- table(z)
  rm <- unique(c(floor((length(z) + 1) / 2), ceiling((length(z) + 1) / 2)))
  cat("Fall", fall, "· Mittelwert:", fmt(mean(z)),
      "· Modalwert(e):", paste(names(hz)[hz == max(hz)], collapse = ", "),
      "· mittlere Kategorien:", paste(z[rm], collapse = ", "),
      "· Medianbereich:", min(z[rm]), "bis", max(z[rm]), "\n")
}


**Rückmeldung:** Im Fall A zieht eine Antwort weit rechts den Mittelwert nach oben (3,8 statt Median 2). Im Fall B sind 2 und 5 beide Modalwerte, der Medianbereich reicht von 2 bis 5; der numerische Median ist 3,5. Im Fall C zählen zwei gültige Antworten, beide mit Kategorie 3. NA ist keine politische Mitte und keine Null.

## 6. Von der Beschreibung zur Hypothese

Wie in früheren Versionen dieses Colabs können wir nach einem möglichen **Gender Gap** fragen: Unterscheiden sich die Links-Rechts-Antworten der beiden hier ausgewiesenen Geschlechtsgruppen?

Formulieren Sie vorab eine Erwartung. Die nächste Tabelle zeigt die Fallzahlen und Mittelwerte; die Grafik vergleicht die **Anteile innerhalb jeder Gruppe**. Wegen der diskreten Skala verwenden wir Balken. Unterdrückte und fehlende Angaben werden für diesen Vergleich ausdrücklich ausgeschlossen. Beschreiben Sie zuerst die sichtbaren Unterschiede; eine kausale Erklärung folgt daraus noch nicht.

In [ ]:
gender_daten <- survey |>
  filter(geschlecht %in% c("Weiblich", "Männlich"), !is.na(lire_self))
gender_daten |>
  group_by(geschlecht) |>
  summarise(n = n(), Mittelwert = mean(lire_self),
            numerischer_Median = median(lire_self), .groups = "drop")
if (nrow(gender_daten) > 0) {
  gender_verteilung <- gender_daten |>
    count(geschlecht, lire_self) |>
    complete(geschlecht, lire_self = 1:11, fill = list(n = 0L)) |>
    group_by(geschlecht) |>
    mutate(Anteil = n / sum(n)) |>
    ungroup()
  p_gender <- ggplot(gender_verteilung, aes(x = lire_self, y = Anteil)) +
    geom_col(fill = "#657A86", width = 0.7) +
    facet_wrap(vars(geschlecht), ncol = 1) +
    scale_x_continuous(breaks = 1:11) +
    scale_y_continuous(labels = scales::label_percent(accuracy = 1)) +
    labs(title = "Links-Rechts-Antworten nach Geschlecht",
         x = "1 = links, 11 = rechts", y = "Anteil innerhalb der Gruppe",
         caption = paste(nrow(gender_daten), "auswertbare Interviews; fehlende und unterdrückte Angaben ausgeschlossen.")) +
    theme_minimal(base_size = 13)
  options(repr.plot.width = 10, repr.plot.height = 6)
  print(p_gender)
}


## 7. Politik hat mehr als eine Achse

Hängen Einstellungen zum Sozialstaat und zum Zuzug zusammen? Jeder Punkt steht für **eine Antwortkombination**; größere Punkte bedeuten mehr Interviews mit dieser Kombination. Die Zahlen bleiben auf ihren tatsächlichen Kategorien. Interviews mit einer fehlenden Antwort werden nur für diese Grafik ausgeschlossen.

Beachten Sie die unterschiedlichen Inhalte der Achsen: Die Skala zum Zuzug ist keine zweite allgemeine Links-Rechts-Selbsteinstufung.

In [ ]:
kombinationen <- survey |>
  filter(!is.na(econ_self), !is.na(immi_self)) |>
  count(econ_self, immi_self)
p_dimensionen <- ggplot(kombinationen, aes(x = econ_self, y = immi_self, size = n)) +
  geom_point(color = "#9C1C26", alpha = 0.65) +
  scale_size_area(max_size = 13,
                  breaks = function(limits) unique(round(seq(max(1, limits[1]), limits[2], length.out = 3)))) +
  scale_x_continuous(breaks = 1:11, limits = c(0.5, 11.5)) +
  scale_y_continuous(breaks = 1:11, limits = c(0.5, 11.5)) +
  labs(title = "Sozialstaat und Zuzug: zwei politische Dimensionen",
       x = "1 = mehr Sozialstaat, 11 = weniger Sozialstaat",
       y = "1 = Zuzug erleichtern, 11 = Zuzug einschränken",
       size = "Anzahl",
       caption = paste("Basis:", sum(kombinationen$n), "Interviews mit beiden Antworten.")) +
  theme_minimal(base_size = 13)
options(repr.plot.width = 10, repr.plot.height = 6)
print(p_dimensionen)


## 8. Jetzt Sie

Wählen Sie eine der anderen Positionsskalen und vergleichen Sie sie mit der Links-Rechts-Selbsteinstufung. Ändern Sie unten zum Beispiel `klim_self` in `euin_self`. Bestimmen Sie zunächst Häufigkeiten und Modalwert(e), anschließend Mittelwert und mittlere Kategorien mit den Befehlen aus Abschnitt 4.

**Zum Abschluss, ohne zurückzublättern:** Erklären Sie einer anderen Person den Unterschied zwischen Modalwert, Median und Mittelwert. Nennen Sie eine Annahme des Mittelwerts und eine Grenze der Kursumfrage.

In [ ]:
survey |>
  count(klim_self) |>
  arrange(klim_self)

# Hier ist Platz für Ihre eigene Auswertung.
